In [108]:
! pip install PILImage

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


ERROR: Could not find a version that satisfies the requirement PILImage (from versions: none)
ERROR: No matching distribution found for PILImage


In [41]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import gdown
from transformers import AutoTokenizer
import torch
from tqdm import tqdm
from stop_words import get_stop_words
from collections import defaultdict
import seaborn as sns
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix,ConfusionMatrixDisplay
import requests
from PIL import Image
from io import BytesIO
import ast
import concurrent.futures
import pickle as pkl
import os
from sklearn.model_selection import train_test_split
tqdm.pandas()

In [8]:
all_samp_test_tsv_fid = '1gPSV95sL7RU_vEG4-I9sp_rguqrDA8jL'
all_samp_val_tsv_fid = '1C5_2_2M_6r7mtQJulrXizjAb95vxRibr'



samp_test_tsv_url = f'https://drive.google.com/uc?id={all_samp_test_tsv_fid}'
samp_val_tsv_url = f'https://drive.google.com/uc?id={all_samp_val_tsv_fid}'


#gdown.download(samp_test_tsv_url, 'all_test_public.tsv')
#gdown.download(samp_val_tsv_url, 'all_validate.tsv')

In [9]:
test_all_df = pd.read_csv('all_test_public.tsv',sep='\t')
val_all_df = pd.read_csv('all_validate.tsv',sep='\t')


print("--------------test data----------------")
display(test_all_df.head(5))
print("--------------validate data----------------")
display(val_all_df.head(5))

--------------test data----------------


,Unnamed: 0.2,Unnamed: 0,Unnamed: 0.1,Unnamed: 0.1.1,author,clean_title,created_utc,domain,hasImage,id,image_url,linked_submission_id,num_comments,score,subreddit,title,upvote_ratio,2_way_label,3_way_label,6_way_label
0,0,0,NaN,NaN,buzzly6,virginia first lady criticized for handing cot...,1.551316e+09,philly.com,False,avkxum,NaN,NaN,2.0,16,nottheonion,Virginia first lady criticized for handing cot...,0.69,1,0,0
1,1,1,NaN,NaN,NaN,jason falconer reluctant hero stopped mall att...,1.474326e+09,cnn.it,False,53kdz3,NaN,NaN,0.0,7,upliftingnews,Jason Falconer: Reluctant hero stopped mall at...,0.77,1,0,0
2,2,2,75691.0,335711.0,myternity,NaN,1.497067e+09,NaN,True,diperbi,http://i.imgur.com/cSIuEVF.jpg,6gc83w,NaN,3,psbattle_artwork,NaN,NaN,0,2,4
3,3,3,NaN,NaN,NaN,woman bites camels testicles to save herself a...,1.569178e+09,wbrz.com,False,d7upss,NaN,NaN,3.0,7,nottheonion,Woman bites camel’s testicles to save herself ...,1.00,1,0,0
4,4,4,190207.0,862707.0,trustbytrust,stargazer,1.425139e+09,NaN,True,cozywbv,http://i.imgur.com/BruWKDi.jpg,2xct9d,NaN,3,psbattle_artwork,stargazer,NaN,0,2,4


--------------validate data----------------


,Unnamed: 0.2,Unnamed: 0,Unnamed: 0.1,Unnamed: 0.1.1,author,clean_title,created_utc,domain,hasImage,id,image_url,linked_submission_id,num_comments,score,subreddit,title,upvote_ratio,2_way_label,3_way_label,6_way_label
0,0,0,71235.0,316517.0,xprmntng,NaN,1.499964e+09,NaN,True,dk69gv8,https://i.imgur.com/snZdHrF.jpg,6n1oet,NaN,505,psbattle_artwork,NaN,NaN,0,2,4
1,1,1,NaN,NaN,singingdart7854,my xbox controller says hi,1.567436e+09,i.redd.it,True,cypw96,https://preview.redd.it/l0ga0tug17k31.jpg?widt...,NaN,4.0,25,mildlyinteresting,My Xbox controller says hi,0.72,1,0,0
2,2,2,NaN,NaN,mandal0re,new image from the mandalorian,1.567745e+09,i.imgur.com,True,d0bzlq,https://external-preview.redd.it/VX7bXDu9Gl8UZ...,NaN,5.0,21,photoshopbattles,PsBattle: New image from The Mandalorian,0.92,1,0,0
3,3,3,126554.0,582609.0,HE_WHO_DRUELS,say hello to my little friend,1.461468e+09,NaN,True,d2ezoob,http://i.imgur.com/F1Zbl3D.jpg,4g6bp9,NaN,10,psbattle_artwork,Say hello to my little friend!,NaN,0,2,4
4,4,4,228704.0,1014196.0,eNaRDe,watch your step little one,1.408047e+09,NaN,True,cjqctpw,http://i.imgur.com/KRyMjn1.jpg,2diyh3,NaN,1,psbattle_artwork,Watch your step little one,NaN,0,2,4


In [10]:
# As far as data cleaning goes the first issue is that they all have these "unamed"
# columns for which those columns need to be dropped.
drop_cols = ['Unnamed: 0.2','Unnamed: 0','Unnamed: 0.1','Unnamed: 0.1.1']

for df in [test_all_df,val_all_df]:
  df.drop(drop_cols,axis=1,inplace=True)


print("--------------test data----------------")
display(test_all_df.head(5))
print("--------------validate data----------------")
display(val_all_df.head(5))

--------------test data----------------


,author,clean_title,created_utc,domain,hasImage,id,image_url,linked_submission_id,num_comments,score,subreddit,title,upvote_ratio,2_way_label,3_way_label,6_way_label
0,buzzly6,virginia first lady criticized for handing cot...,1.551316e+09,philly.com,False,avkxum,NaN,NaN,2.0,16,nottheonion,Virginia first lady criticized for handing cot...,0.69,1,0,0
1,NaN,jason falconer reluctant hero stopped mall att...,1.474326e+09,cnn.it,False,53kdz3,NaN,NaN,0.0,7,upliftingnews,Jason Falconer: Reluctant hero stopped mall at...,0.77,1,0,0
2,myternity,NaN,1.497067e+09,NaN,True,diperbi,http://i.imgur.com/cSIuEVF.jpg,6gc83w,NaN,3,psbattle_artwork,NaN,NaN,0,2,4
3,NaN,woman bites camels testicles to save herself a...,1.569178e+09,wbrz.com,False,d7upss,NaN,NaN,3.0,7,nottheonion,Woman bites camel’s testicles to save herself ...,1.00,1,0,0
4,trustbytrust,stargazer,1.425139e+09,NaN,True,cozywbv,http://i.imgur.com/BruWKDi.jpg,2xct9d,NaN,3,psbattle_artwork,stargazer,NaN,0,2,4


--------------validate data----------------


,author,clean_title,created_utc,domain,hasImage,id,image_url,linked_submission_id,num_comments,score,subreddit,title,upvote_ratio,2_way_label,3_way_label,6_way_label
0,xprmntng,NaN,1.499964e+09,NaN,True,dk69gv8,https://i.imgur.com/snZdHrF.jpg,6n1oet,NaN,505,psbattle_artwork,NaN,NaN,0,2,4
1,singingdart7854,my xbox controller says hi,1.567436e+09,i.redd.it,True,cypw96,https://preview.redd.it/l0ga0tug17k31.jpg?widt...,NaN,4.0,25,mildlyinteresting,My Xbox controller says hi,0.72,1,0,0
2,mandal0re,new image from the mandalorian,1.567745e+09,i.imgur.com,True,d0bzlq,https://external-preview.redd.it/VX7bXDu9Gl8UZ...,NaN,5.0,21,photoshopbattles,PsBattle: New image from The Mandalorian,0.92,1,0,0
3,HE_WHO_DRUELS,say hello to my little friend,1.461468e+09,NaN,True,d2ezoob,http://i.imgur.com/F1Zbl3D.jpg,4g6bp9,NaN,10,psbattle_artwork,Say hello to my little friend!,NaN,0,2,4
4,eNaRDe,watch your step little one,1.408047e+09,NaN,True,cjqctpw,http://i.imgur.com/KRyMjn1.jpg,2diyh3,NaN,1,psbattle_artwork,Watch your step little one,NaN,0,2,4


In [12]:
print('------------Test set na cnt---------------------')
print(test_all_df.isna().sum())
print('------------dev set na cnt---------------------')
print(val_all_df.isna().sum())

------------Test set na cnt---------------------
author                  15692
clean_title              7963
created_utc                 0
domain                  25311
hasImage                    0
id                          0
image_url               25345
linked_submission_id    67133
num_comments            25311
score                       0
subreddit                   0
title                    7790
upvote_ratio            25311
2_way_label                 0
3_way_label                 0
6_way_label                 0
dtype: int64
------------dev set na cnt---------------------
author                  15677
clean_title              7908
created_utc                 0
domain                  25587
hasImage                    0
id                          0
image_url               25409
linked_submission_id    66857
num_comments            25587
score                       0
subreddit                   0
title                    7723
upvote_ratio            25587
2_way_label         

In [15]:
cols = ['author','clean_title','id','image_url','2_way_label','3_way_label','6_way_label']


test_all_df = test_all_df[cols].copy().dropna()
val_all_df = val_all_df[cols].copy().dropna()


print(test_all_df.shape)
print(val_all_df.shape)

(56097, 7)
(56107, 7)


In [16]:
tokenizer = AutoTokenizer.from_pretrained('google-bert/bert-base-uncased')

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [23]:
english_stop_word_ids = tokenizer(' '.join(get_stop_words('en')),add_special_tokens=True,return_tensors='np')['input_ids'][0]

def get_bert_toks(sent):

  tok_ids_wspec = tokenizer(sent,add_special_tokens=False,return_tensors='np')['input_ids'][0]

  tok_ids = np.setdiff1d(tok_ids_wspec,english_stop_word_ids)

  return tokenizer.convert_ids_to_tokens(tok_ids)

In [29]:
# We are gonna first focus on visualizations of our train set.
test_all_df['text_toks'] = test_all_df['clean_title'].progress_apply(get_bert_toks)
val_all_df['text_toks'] = val_all_df['clean_title'].progress_apply(get_bert_toks)

100%|███████████████████████████████████████████████████████████████████████████| 56107/56107 [00:04<00:00, 11225.09it/s]


In [54]:
# Global Session to prevent SSL handshake throttling
session = requests.Session()
session.headers.update({'User-Agent': 'Mozilla/5.0'})

def process_single_image(row):
    """This is the isolated function each thread will run."""
    url = row['image_url']
    
    try:
        if pd.isna(url):
            return None # Failed
            
        response = session.get(url, timeout=3)
        if response.status_code == 200:
            img = PILImage.open(io.BytesIO(response.content)).convert('RGB')
            img_array = np.array(img)

            # RGB totals (fast version)
            tot_r = np.sum(img_array[:, :, 0])
            tot_g = np.sum(img_array[:, :, 1])
            tot_b = np.sum(img_array[:, :, 2])

            total_rgb = tot_r + tot_g + tot_b
            if total_rgb == 0:
                return None # Failed

            r_perc = (tot_r / total_rgb) * 100
            g_perc = (tot_g / total_rgb) * 100
            b_perc = (tot_b / total_rgb) * 100

            new_row = row.copy()
            new_row['rgb_img_vals'] = (r_perc, g_perc, b_perc)
            return new_row # Passed
            
        return None # Failed (Status code not 200)
    except Exception:
        return None # Failed (Timeout, bad image data, etc.)


def create_clean_df_fast(source_df, in_progress_rows=None, max_workers=20, pass_cnt_pth=None, fail_cnt_pth=None):
    passed = 0
    failed = 0
    
    # Load previous counts if provided
    if pass_cnt_pth is not None and fail_cnt_pth is not None:
        with open('failed_cnt.pkl','rb') as f:
            failed = pkl.load(f)
        with open('passed_cnt.pkl','rb') as f:
            passed = pkl.load(f)

    valid_rows = []
    if in_progress_rows is not None and not in_progress_rows.empty:
        valid_rows = in_progress_rows.to_dict('records')
        processed_ids = set(in_progress_rows['id'])
        before = len(source_df)
        source_df = source_df[~source_df['id'].isin(processed_ids)].copy()
        after = len(source_df)
        print(f"Removed {before - after} already processed rows")
        print(f"Remaining rows to process: {after}")
        
    total = len(source_df)
    rows_to_process = source_df.to_dict('records')

    print(f"Starting ThreadPoolExecutor with {max_workers} workers...")
    
    # Spin up the threading environment cleanly
    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_to_row = {executor.submit(process_single_image, row): row for row in rows_to_process}
        pbar = tqdm(concurrent.futures.as_completed(future_to_row), total=total, desc="Processing images")
        
        for future in pbar:
            result = future.result()
            
            if result is not None:
                valid_rows.append(result)
                passed += 1
                
                # Checkpoint saving block (Set to 200 as you requested)
                if passed % 50 == 0:
                    print(f"Saving {len(valid_rows)} rows and counts to disk...")
                    pd.DataFrame(valid_rows).to_pickle("inprogress.pkl")
                    with open('failed_cnt.pkl','wb') as f:
                        pkl.dump(failed, f)
                    with open('passed_cnt.pkl','wb') as f:
                        pkl.dump(passed, f)
            else:
                failed += 1

            pbar.set_postfix(Passed=passed, Failed=failed)

    # Final wrap-up
    clean_df = pd.DataFrame(valid_rows)
    np.save('test_rbgavg_vals.npy', np.array(list(clean_df['rgb_img_vals'])))

    print(f"Total rows: {total}")
    print(f"Successfully processed: {passed}")
    print(f"Failed: {failed}")
    if total > 0:
        print(f"Success rate: {(passed / total) * 100:.2f}%")

    return clean_df

In [55]:
create_clean_df(test_all_df)

Processing images: 100%|█████████████████████████████████████████████████████████| 56097/56097 [3:49:04<00:00,  4.08it/s]


KeyError: 'rgb_img_vals'

In [8]:
quant_drive_pth = '/Users/tylergarfield/Desktop/school/CSC480/CSC_480_Project_Pratt-Garfield/images'
os.makedirs(quant_drive_pth,exist_ok=True)
session = requests.Session()
session.headers.update({'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'})

def download_and_sanitize_image(img_id,url):
    
    try:
        if pd.isna(url):
            return None
        
        response = session.get(url, timeout=4)
        if response.status_code == 200:
            # NOTE GEMINI SCREWED THIS UP AT FIRST THIS IS WHY WE STILL NEED HUMANS FOR COING!!!!
            img = Image.open(BytesIO(response.content)).convert('RGB')
            
            # 2. Define the exact save path on your quarantine drive
            save_path = os.path.join(quant_drive_pth, f"{img_id}.jpg")
            
            # 3. Sanitize and Save! 
            # This writes a clean, brand new file to the drive, leaving malware behind
            img.save(save_path, format="JPEG", quality=90)
            
            return img_id # Passed
            
        return None 
    except Exception as e:
        print(e)
        return None

In [10]:
failed_ids = []

with open('photo_download_last.pkl','wb') as f:
    pkl.dump(0,f)
with open('photo_failed_ids.pkl','wb') as f:
    pkl.dump([],f)

In [20]:
model_data_clean = pd.read_csv('train_dat/fake_news_traindf.csv')

downloaded_photos_inds = []
failed_ids = []

downloaded_photos_last = 0
with open('photo_download_last.pkl','rb') as f:
    downloaded_photos_last = pkl.load(f)
with open('photo_failed_ids.pkl','rb') as f:
    failed_ids = pkl.load(f)

if downloaded_photos_last != 0:
    downloaded_photos_inds = np.arange(0,(downloaded_photos_last+1))

images_to_retrive = model_data_clean.drop(index=downloaded_photos_inds)


imgs_needed = images_to_retrive.to_dict('records')


try:
    pbar = tqdm(imgs_needed,total=len(imgs_needed),leave=True)
    pbar.set_description(f'last ind {downloaded_photos_last}; Number of failed images is {len(failed_ids)} ❌')
    for data_pt in pbar:
        img_id_curr = data_pt['id']
        img_url_curr = data_pt['image_url']
    
    
        ret_photo_id = download_and_sanitize_image(img_id_curr,img_url_curr)
    
        if ret_photo_id is None:
            #print(f'Photo download for file with id {img_id_curr} failed ❌!')
            failed_ids.append(img_id_curr)
            
        
        with open('photo_download_last.pkl','wb') as f:
            pkl.dump(downloaded_photos_last,f)

        pbar.set_description(f'last ind {downloaded_photos_last}; Number of failed images is {len(failed_ids)} ❌')
    
        downloaded_photos_last += 1
    
        

except KeyboardInterrupt:
    #print('Saving the last index of last saved file now! ✅')
    with open('photo_download_last.pkl','wb') as f:
            pkl.dump(downloaded_photos_last,f)
    with open('photo_failed_ids.pkl','wb') as f:
            pkl.dump(failed_ids,f)

last ind 33288; Number of failed images is 61 ❌:   0%|                           | 4/286714 [00:04<114:40:29,  1.44s/it]]

HTTPSConnectionPool(host='preview.redd.it', port=443): Read timed out. (read timeout=4)


last ind 34902; Number of failed images is 61 ❌:   1%|▏                        | 1618/286714 [04:21<13:32:35,  5.85it/s]/Users/tylergarfield/gemini_chats/new_mps_env/lib/python3.11/site-packages/PIL/Image.py:1039: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
last ind 72286; Number of failed images is 64 ❌:  14%|███▏                   | 39002/286714 [1:34:55<7:27:43,  9.22it/s]/Users/tylergarfield/gemini_chats/new_mps_env/lib/python3.11/site-packages/PIL/Image.py:3432: DecompressionBombWarning: Image size (110718270 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(
last ind 89393; Number of failed images is 68 ❌:  20%|████▎                 | 56109/286714 [2:14:27<69:49:38,  1.09s/it]

HTTPSConnectionPool(host='preview.redd.it', port=443): Read timed out. (read timeout=4)


last ind 121956; Number of failed images is 70 ❌:  31%|██████▍              | 88671/286714 [3:29:28<47:33:58,  1.16it/s]

HTTPSConnectionPool(host='nodincaldain.tripod.com', port=443): Read timed out. (read timeout=4)


last ind 127450; Number of failed images is 70 ❌:  33%|███████▏              | 94166/286714 [3:42:03<7:34:02,  7.07it/s]


In [7]:
test = model_data_clean.to_dict('records')[13999]
#response = session.get(test['image_url'], timeout=4)
#Image.open(BytesIO(response.content)).convert('RGB')
print(download_and_sanitize_image(test['id'],test['image_url']))

c7jn63y


In [31]:
img_path = '/Users/tylergarfield/Desktop/school/CSC480/CSC_480_Project_Pratt-Garfield/images'
img_files = os.listdir(img_path)
img_ids = [img_name.strip('.jpg') for img_name in img_files]

model_data_clean[model_data_clean['id'].isin(img_ids)]

,author,clean_title,id,image_url,2_way_label,3_way_label,6_way_label,text_toks
0,Alexithymia,my walgreens offbrand mucinex was engraved wit...,awxhir,https://external-preview.redd.it/WylDbZrnbvZdB...,1,0,0,"['##ns', 'letters', 'engraved', '##cine', '##b..."
1,prometheus1123,hackers leak emails from uae ambassador to us,6f2cy5,https://external-preview.redd.it/6fNhdbc6K1vFA...,1,0,0,"['ambassador', 'leak', 'uae', 'emails', 'hacker']"
3,happenpupe,major thermos,6d50rl,https://preview.redd.it/l9gvkkf3jizy.jpg?width...,0,2,2,"['major', '##rm']"
4,nyswagggggggg,rabbi meat from cloned pig could be kosher for...,86byl8,https://external-preview.redd.it/KHisCPOGwz7cz...,1,0,0,"['eat', 'jews', 'meat', 'milk', 'rabbi', 'pig'..."
5,ApiContraption,cutouts,cgp0lmq,http://i.imgur.com/vbveIEd%2ejpg,0,2,4,"['cut', '##outs']"
...,...,...,...,...,...,...,...,...
127448,atthecorner,buy a share in america s,9133cd,https://external-preview.redd.it/XXrtr-at5pomE...,0,1,5,"['america', 'share']"
127449,metatron121,dog on a bed,bl3jtq,https://preview.redd.it/oo408n4nrgw21.jpg?widt...,1,0,0,"['bed', 'dog']"
127450,thenickh,prince charles at a sword dance in saudi arabia,1yads7,https://external-preview.redd.it/CmcWSD6X5AEsn...,1,0,0,"['charles', 'dance', 'prince', 'sword', 'saudi..."
127451,Rotorammer,the only strayan thing in the room,czgg901,http://i.imgur.com/OYxlsyF.jpg,0,2,4,"['##an', 'stray']"


In [38]:
len(img_ids)-len(set(img_ids))

3

In [40]:
def get_duplicates(items):
    seen = set()
    dupes = set()
    for x in items:
        if x in seen:
            dupes.add(x)
        else:
            seen.add(x)
    return list(dupes)

get_duplicates(img_ids)

['cjaihb', 'ctazc3', 'ayv3w']

In [58]:
model_data_clean = pd.read_csv('train_dat/fake_news_traindf.csv')
model_data_clean = model_data_clean[model_data_clean['id'].isin(img_ids)].copy()
model_data_clean.drop(columns=['text_toks'])

df_temp, df_test = train_test_split(model_data_clean,train_size=0.8,random_state=42)
df_train, df_dev = train_test_split(df_temp,train_size=0.8,random_state=42)

df_train.reset_index(drop=True,inplace=True)
df_test.reset_index(drop=True,inplace=True)
df_dev.reset_index(drop=True,inplace=True)

print(df_train.shape)
print(df_test.shape)
print(df_dev.shape)

(74727, 8)
(23353, 8)
(18682, 8)


In [59]:
partition_dfs = [df_train,df_test,df_dev]
partition_paths = ['train','test','dev']
dat_path = '/Users/tylergarfield/Desktop/school/CSC480/CSC_480_Project_Pratt-Garfield/data'



for i in range(3):

    print(f"On {partition_paths[i]}")

    curr_df = partition_dfs[i]
    drop_inds_curr = []

    
    for ind,row in curr_df.iterrows():
        img_id = row['id']
        source = f'{img_path}/{img_id}.jpg'

        if not os.path.exists(source):
            drop_inds_curr.append(ind)
            print("Found a row who's supposed image did not exist!! Dropping row")
            continue
            
                
        dest = f'{dat_path}/{partition_paths[i]}/{img_id}.jpg'

        # Move image to proper partition.
        os.rename(source,dest)

    curr_df = curr_df.drop(index=drop_inds_curr).reset_index(drop=True)

On train
Found a row who's supposed image did not exist!! Dropping row
On test
On dev


In [60]:
print(df_train.shape)
print(df_test.shape)
print(df_dev.shape)

(74727, 8)
(23353, 8)
(18682, 8)


In [73]:
drop_rows = []

for ind,row in df_dev.iterrows():
    img_id = row['id']
    img_path = f'{dat_path}/dev/{img_id}.jpg'
    if not os.path.exists(img_path):
        print("Found entry with no image!")
        #drop_rows.append(ind)

#df_train = df_train.drop(index=drop_rows)
print(df_train.shape)
print(df_test.shape)
print(df_dev.shape)

(74726, 8)
(23353, 8)
(18682, 8)


In [78]:
df_train.to_csv('train_df_clean.csv',index=False)
df_test.to_csv('test_df_clean.csv',index=False)
df_dev.to_csv('dev_df_clean.csv',index=False)

In [77]:
df_train.drop(columns=['text_toks','image_url'],inplace=True)
df_test.drop(columns=['text_toks','image_url'],inplace=True)
df_dev.drop(columns=['text_toks','image_url'],inplace=True)